# MoE Live Demo: Watch the Experts Light Up

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prashantkul/learn-generative-ai/blob/main/10-mixture-of-experts/live-moe-demo.ipynb)

**GPU recommended:** Yes (model inference is faster on GPU). Use Colab's free T4.

This notebook loads a real MoE model and visualizes expert routing in action.
We'll see which experts activate for different tokens and input types.

---

## 1. Setup and Model Loading

In [ ]:
!pip install -q transformers accelerate torch bitsandbytes sentencepiece seaborn scipy scikit-learn

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import defaultdict
from scipy.stats import entropy as scipy_entropy
from sklearn.metrics.pairwise import cosine_similarity

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "figure.dpi": 120,
})

sns.set_palette("husl")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen1.5-MoE-A2.7B-Chat"
FALLBACK_MODEL_ID = "mistralai/Mixtral-8x7B-v0.1"

model = None
tokenizer = None
model_name = None

# Attempt 1: Load Qwen1.5-MoE-A2.7B-Chat in float16
try:
    print(f"Loading {MODEL_ID} ...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    model_name = MODEL_ID
    print(f"Successfully loaded {MODEL_ID}")
except Exception as e:
    print(f"Failed to load {MODEL_ID}: {e}")
    print(f"\nFalling back to {FALLBACK_MODEL_ID} with 4-bit quantization ...")
    try:
        from transformers import BitsAndBytesConfig
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
        )
        tokenizer = AutoTokenizer.from_pretrained(FALLBACK_MODEL_ID)
        model = AutoModelForCausalLM.from_pretrained(
            FALLBACK_MODEL_ID,
            quantization_config=bnb_config,
            device_map="auto",
        )
        model_name = FALLBACK_MODEL_ID
        print(f"Successfully loaded {FALLBACK_MODEL_ID} (4-bit)")
    except Exception as e2:
        raise RuntimeError(
            f"Could not load either model. "
            f"Primary: {e}. Fallback: {e2}. "
            f"Try using a GPU runtime with more memory."
        )

model.eval()
print(f"\nModel: {model_name}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

In [ ]:
# Print a summary of the model architecture
print("Model architecture (top-level modules):")
print("=" * 60)
for name, module in model.named_children():
    num_params = sum(p.numel() for p in module.parameters()) / 1e6
    print(f"  {name}: {module.__class__.__name__} ({num_params:.1f}M params)")

print("\nFirst transformer layer structure:")
print("-" * 60)
first_layer = list(model.model.layers)[0]
for name, module in first_layer.named_children():
    num_params = sum(p.numel() for p in module.parameters()) / 1e6
    print(f"  {name}: {module.__class__.__name__} ({num_params:.1f}M params)")

---

## 2. Inspect the MoE Architecture

We identify all MoE layers in the model, locate the router (gate) modules,
and print key parameters: number of experts, active experts per token, and
expert dimensions.

In [ ]:
def find_moe_layers(model):
    """
    Walk the model and find all MoE layers.
    Supports Qwen1.5-MoE (Qwen2MoeSparseMoeBlock) and Mixtral (MixtralSparseMoeBlock).
    Returns a list of (layer_index, moe_module, gate_module) tuples.
    """
    moe_layers = []

    for layer_idx, layer in enumerate(model.model.layers):
        mlp = layer.mlp
        class_name = mlp.__class__.__name__

        # Check if this is a sparse MoE block (vs a dense MLP)
        if "SparseMoe" in class_name or "MoE" in class_name:
            gate = getattr(mlp, "gate", None)
            if gate is not None:
                moe_layers.append((layer_idx, mlp, gate))
        elif hasattr(mlp, "gate"):
            moe_layers.append((layer_idx, mlp, mlp.gate))

    return moe_layers


moe_layers_info = find_moe_layers(model)
print(f"Found {len(moe_layers_info)} MoE layers out of {len(list(model.model.layers))} total layers")
print()

In [ ]:
# Examine the first MoE layer in detail
if moe_layers_info:
    layer_idx, moe_block, gate = moe_layers_info[0]

    print(f"MoE block class: {moe_block.__class__.__name__}")
    print(f"Gate class:      {gate.__class__.__name__}")
    print()

    # Gate weight shape reveals (num_experts, d_model)
    gate_weight = gate.weight
    num_experts_total = gate_weight.shape[0]
    d_model = gate_weight.shape[1]
    print(f"Gate weight shape: {gate_weight.shape}")
    print(f"  -> Number of experts: {num_experts_total}")
    print(f"  -> Hidden dimension (d_model): {d_model}")
    print()

    # Determine top-k from model config
    config = model.config
    top_k = getattr(config, "num_experts_per_tok", None)
    if top_k is None:
        top_k = getattr(config, "num_selected_experts", 2)
    print(f"Active experts per token (top-k): {top_k}")
    print(f"Total experts: {num_experts_total}")
    print(f"Sparsity: only {top_k}/{num_experts_total} = {100*top_k/num_experts_total:.1f}% of experts active per token")
    print()

    # Show expert structure
    experts_container = getattr(moe_block, "experts", None)
    if experts_container is not None:
        if hasattr(experts_container, '__len__'):
            first_expert = experts_container[0]
        else:
            first_expert = list(experts_container.children())[0]
        print(f"Expert structure ({first_expert.__class__.__name__}):")
        for name, param in first_expert.named_parameters():
            print(f"  {name}: {param.shape}")
        expert_params = sum(p.numel() for p in first_expert.parameters())
        print(f"  Parameters per expert: {expert_params / 1e6:.2f}M")
        print(f"  Active expert params per token: {expert_params * top_k / 1e6:.2f}M")

    # Check for shared expert
    shared_expert = getattr(moe_block, "shared_expert", None)
    if shared_expert is not None:
        shared_params = sum(p.numel() for p in shared_expert.parameters())
        print(f"\nShared expert: {shared_expert.__class__.__name__} ({shared_params / 1e6:.2f}M params)")
        print("  (The shared expert processes ALL tokens, in addition to the routed experts)")

In [ ]:
# Summary table of all MoE layers and their gate shapes
print(f"{'Layer':>6}  {'Gate Shape':>20}  {'Experts':>8}  {'Type':>15}")
print("-" * 55)
for layer_idx, moe_block, gate in moe_layers_info:
    shape = tuple(gate.weight.shape)
    n_exp = shape[0]
    block_type = moe_block.__class__.__name__
    print(f"{layer_idx:>6}  {str(shape):>20}  {n_exp:>8}  {block_type:>15}")

---

## 3. Hook into the Router

We register forward hooks on every gate (router) module to intercept the routing
decisions as they happen. For each layer and each token, we capture:
- **Router logits**: the raw (pre-softmax) scores for every expert
- **Router weights**: the post-softmax probabilities
- **Selected experts**: the top-k expert indices chosen for each token

In [ ]:
class RoutingCapture:
    """Captures router decisions from all MoE layers via forward hooks."""

    def __init__(self, model, moe_layers_info, top_k):
        self.model = model
        self.top_k = top_k
        self.hooks = []
        self.captured = {}  # layer_idx -> {logits, probs, top_k_indices, top_k_weights}

        for layer_idx, moe_block, gate in moe_layers_info:
            hook = gate.register_forward_hook(self._make_hook(layer_idx))
            self.hooks.append(hook)

    def _make_hook(self, layer_idx):
        def hook_fn(module, input, output):
            # output is the gate logits: (batch, seq_len, num_experts)
            # For some models, the gate output may be just logits (a single tensor)
            if isinstance(output, tuple):
                logits = output[0]
            else:
                logits = output

            logits_detached = logits.detach().float().cpu()
            probs = torch.softmax(logits_detached, dim=-1)
            top_k_weights, top_k_indices = torch.topk(probs, self.top_k, dim=-1)

            self.captured[layer_idx] = {
                "logits": logits_detached,
                "probs": probs,
                "top_k_indices": top_k_indices,
                "top_k_weights": top_k_weights,
            }
        return hook_fn

    def clear(self):
        self.captured.clear()

    def remove_hooks(self):
        for hook in self.hooks:
            hook.remove()
        self.hooks.clear()

    def get_routing_data(self):
        """Return captured data sorted by layer index."""
        return dict(sorted(self.captured.items()))


capture = RoutingCapture(model, moe_layers_info, top_k)
print(f"Registered {len(capture.hooks)} forward hooks on router/gate modules")
print(f"Top-k: {top_k}")

---

## 4. Run Inference and Capture Routing

We run four different types of prompts through the model and capture all routing
decisions. Each prompt exercises a different kind of knowledge.

In [ ]:
prompts = {
    "math": "What is 2 + 2?",
    "code": "def fibonacci(n):",
    "creative": "Once upon a time in a dark forest,",
    "factual": "The capital of France is",
}

all_routing_data = {}  # prompt_name -> {layer_idx -> routing data}
all_tokens = {}        # prompt_name -> list of token strings

for prompt_name, prompt_text in prompts.items():
    print(f"\nProcessing [{prompt_name}]: \"{prompt_text}\"")

    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    token_ids = inputs["input_ids"][0].tolist()
    token_strings = [tokenizer.decode([tid]) for tid in token_ids]
    all_tokens[prompt_name] = token_strings

    capture.clear()
    with torch.no_grad():
        outputs = model(**inputs)

    all_routing_data[prompt_name] = capture.get_routing_data()

    # Print a summary of routing for the first MoE layer
    first_moe_idx = list(all_routing_data[prompt_name].keys())[0]
    data = all_routing_data[prompt_name][first_moe_idx]
    print(f"  Tokens ({len(token_strings)}): {token_strings}")
    print(f"  Top-{top_k} experts at layer {first_moe_idx}: "
          f"{data['top_k_indices'][0].tolist()}")

print("\nRouting data captured for all prompts.")

---

## 5. Visualize Expert Activation Heatmaps

For each prompt, we create a heatmap showing which experts were selected at each
layer for each token. The x-axis is token position, y-axis is MoE layer index,
and color indicates the top-1 selected expert.

In [ ]:
def plot_expert_heatmap(routing_data, tokens, prompt_name, top_k_slot=0):
    """
    Plot a heatmap of expert selections across layers and tokens.

    Args:
        routing_data: dict mapping layer_idx -> routing info
        tokens: list of token strings
        prompt_name: string label for the plot title
        top_k_slot: which top-k slot to show (0 = top-1, 1 = top-2, etc.)
    """
    layer_indices = sorted(routing_data.keys())
    num_layers = len(layer_indices)
    num_tokens = len(tokens)

    # Build the expert selection matrix: (num_layers, num_tokens)
    expert_matrix = np.zeros((num_layers, num_tokens), dtype=int)
    weight_matrix = np.zeros((num_layers, num_tokens))

    for row, layer_idx in enumerate(layer_indices):
        data = routing_data[layer_idx]
        indices = data["top_k_indices"][0, :, top_k_slot].numpy()
        weights = data["top_k_weights"][0, :, top_k_slot].numpy()
        expert_matrix[row, :num_tokens] = indices[:num_tokens]
        weight_matrix[row, :num_tokens] = weights[:num_tokens]

    fig, axes = plt.subplots(1, 2, figsize=(16, max(4, num_layers * 0.25 + 2)),
                             gridspec_kw={"width_ratios": [1, 1]})

    # Left: Expert ID heatmap
    slot_label = f"top-{top_k_slot + 1}"
    im1 = axes[0].imshow(expert_matrix, aspect="auto", cmap="tab20",
                          interpolation="nearest")
    axes[0].set_xlabel("Token position")
    axes[0].set_ylabel("MoE layer index")
    axes[0].set_title(f"{slot_label} expert ID -- {prompt_name}")
    axes[0].set_yticks(range(num_layers))
    axes[0].set_yticklabels(layer_indices, fontsize=7)

    # Token labels on x-axis (truncated for readability)
    truncated = [t[:8] for t in tokens]
    axes[0].set_xticks(range(num_tokens))
    axes[0].set_xticklabels(truncated, rotation=45, ha="right", fontsize=7)
    plt.colorbar(im1, ax=axes[0], label="Expert ID", shrink=0.7)

    # Right: Expert weight (confidence) heatmap
    im2 = axes[1].imshow(weight_matrix, aspect="auto", cmap="YlOrRd",
                          vmin=0, interpolation="nearest")
    axes[1].set_xlabel("Token position")
    axes[1].set_ylabel("MoE layer index")
    axes[1].set_title(f"{slot_label} expert weight -- {prompt_name}")
    axes[1].set_yticks(range(num_layers))
    axes[1].set_yticklabels(layer_indices, fontsize=7)
    axes[1].set_xticks(range(num_tokens))
    axes[1].set_xticklabels(truncated, rotation=45, ha="right", fontsize=7)
    plt.colorbar(im2, ax=axes[1], label="Routing weight", shrink=0.7)

    plt.suptitle(f'"{prompts[prompt_name]}"', fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot top-1 expert heatmaps for all prompts
for prompt_name in prompts:
    plot_expert_heatmap(
        all_routing_data[prompt_name],
        all_tokens[prompt_name],
        prompt_name,
        top_k_slot=0,
    )

In [ ]:
# Plot top-2 expert heatmaps for all prompts
for prompt_name in prompts:
    plot_expert_heatmap(
        all_routing_data[prompt_name],
        all_tokens[prompt_name],
        prompt_name,
        top_k_slot=1,
    )

---

## 6. Expert Utilization Across Prompts

We aggregate routing decisions across all prompts to see which experts are used
most frequently overall, and whether utilization varies by layer.

In [ ]:
def compute_expert_utilization(all_routing_data, num_experts):
    """
    Count how many times each expert was selected across all prompts.
    Returns:
        overall_counts: (num_experts,) array
        per_layer_counts: dict mapping layer_idx -> (num_experts,) array
        per_prompt_counts: dict mapping prompt_name -> (num_experts,) array
    """
    overall_counts = np.zeros(num_experts)
    per_layer_counts = defaultdict(lambda: np.zeros(num_experts))
    per_prompt_counts = {}

    for prompt_name, routing_data in all_routing_data.items():
        prompt_counts = np.zeros(num_experts)
        for layer_idx, data in routing_data.items():
            # data["top_k_indices"]: (1, seq_len, top_k)
            indices = data["top_k_indices"][0].numpy().flatten()
            for eidx in indices:
                if eidx < num_experts:
                    overall_counts[eidx] += 1
                    per_layer_counts[layer_idx][eidx] += 1
                    prompt_counts[eidx] += 1
        per_prompt_counts[prompt_name] = prompt_counts

    return overall_counts, dict(per_layer_counts), per_prompt_counts


overall_counts, per_layer_counts, per_prompt_counts = compute_expert_utilization(
    all_routing_data, num_experts_total
)

print(f"Total expert selections across all prompts: {int(overall_counts.sum())}")
print(f"Number of unique experts used: {np.sum(overall_counts > 0)}")

In [ ]:
# Overall expert utilization bar chart
fig, ax = plt.subplots(figsize=(14, 5))

colors = plt.cm.tab20(np.linspace(0, 1, num_experts_total))
bars = ax.bar(range(num_experts_total), overall_counts, color=colors, edgecolor="gray",
              linewidth=0.5)

# Mark the ideal uniform level
ideal = overall_counts.sum() / num_experts_total
ax.axhline(y=ideal, color="red", linestyle="--", alpha=0.6,
           label=f"Ideal uniform ({ideal:.0f})")

ax.set_xlabel("Expert ID")
ax.set_ylabel("Number of times selected")
ax.set_title("Expert Utilization Across All Prompts")
ax.legend()

# Only show every Nth tick if too many experts
if num_experts_total > 20:
    tick_step = max(1, num_experts_total // 20)
    ax.set_xticks(range(0, num_experts_total, tick_step))
else:
    ax.set_xticks(range(num_experts_total))

plt.tight_layout()
plt.show()

In [ ]:
# Per-layer expert utilization heatmap
sorted_layers = sorted(per_layer_counts.keys())
num_layers_moe = len(sorted_layers)

# Sample layers if there are too many for a readable plot
if num_layers_moe > 20:
    step = max(1, num_layers_moe // 20)
    sampled_layers = sorted_layers[::step]
else:
    sampled_layers = sorted_layers

utilization_matrix = np.zeros((len(sampled_layers), num_experts_total))
for row, layer_idx in enumerate(sampled_layers):
    counts = per_layer_counts[layer_idx]
    total = counts.sum()
    if total > 0:
        utilization_matrix[row] = counts / total  # normalize per layer

fig, ax = plt.subplots(figsize=(14, max(4, len(sampled_layers) * 0.3 + 1)))
im = ax.imshow(utilization_matrix, aspect="auto", cmap="Blues", interpolation="nearest")
ax.set_xlabel("Expert ID")
ax.set_ylabel("MoE Layer Index")
ax.set_title("Per-Layer Expert Utilization (normalized per layer)")
ax.set_yticks(range(len(sampled_layers)))
ax.set_yticklabels(sampled_layers, fontsize=7)
if num_experts_total <= 20:
    ax.set_xticks(range(num_experts_total))
plt.colorbar(im, ax=ax, label="Fraction of tokens", shrink=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Per-prompt expert utilization comparison
fig, ax = plt.subplots(figsize=(14, 5))

prompt_names = list(per_prompt_counts.keys())
num_prompts = len(prompt_names)
bar_width = 0.8 / num_prompts
prompt_colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12"]

# Only show experts that were actually used
used_experts = np.where(overall_counts > 0)[0]
x_positions = np.arange(len(used_experts))

for pidx, pname in enumerate(prompt_names):
    counts = per_prompt_counts[pname][used_experts]
    offset = (pidx - num_prompts / 2 + 0.5) * bar_width
    ax.bar(x_positions + offset, counts, bar_width, label=pname,
           color=prompt_colors[pidx % len(prompt_colors)], alpha=0.8,
           edgecolor="gray", linewidth=0.3)

ax.set_xlabel("Expert ID")
ax.set_ylabel("Number of times selected")
ax.set_title("Expert Utilization by Prompt Type")
ax.set_xticks(x_positions)
ax.set_xticklabels(used_experts, fontsize=7)
ax.legend()
plt.tight_layout()
plt.show()

---

## 7. Token-Level Expert Routing

For a single prompt, we create a detailed view that maps each token to its
selected experts at every MoE layer. This reveals patterns in how the model
routes different types of tokens (punctuation, nouns, verbs, numbers, etc.).

In [ ]:
def plot_token_routing_detail(routing_data, tokens, prompt_name, max_layers=12):
    """
    Show a detailed annotated heatmap of expert routing for a single prompt.
    Each cell shows the top-1 expert ID, colored by expert identity.
    """
    layer_indices = sorted(routing_data.keys())
    if len(layer_indices) > max_layers:
        # Sample evenly
        step = max(1, len(layer_indices) // max_layers)
        layer_indices = layer_indices[::step][:max_layers]

    num_layers = len(layer_indices)
    num_tokens = len(tokens)

    # Build the annotation matrix
    expert_matrix = np.zeros((num_layers, num_tokens), dtype=int)
    annotation_matrix = []

    for row, layer_idx in enumerate(layer_indices):
        data = routing_data[layer_idx]
        row_annotations = []
        for tok in range(num_tokens):
            top1 = data["top_k_indices"][0, tok, 0].item()
            top1_w = data["top_k_weights"][0, tok, 0].item()
            expert_matrix[row, tok] = top1
            if top_k > 1:
                top2 = data["top_k_indices"][0, tok, 1].item()
                row_annotations.append(f"{top1}\n({top2})")
            else:
                row_annotations.append(str(top1))
        annotation_matrix.append(row_annotations)

    fig, ax = plt.subplots(figsize=(max(10, num_tokens * 1.0),
                                     max(4, num_layers * 0.5 + 1)))

    im = ax.imshow(expert_matrix, aspect="auto", cmap="tab20",
                    interpolation="nearest")

    # Annotate each cell
    for row in range(num_layers):
        for col in range(num_tokens):
            ax.text(col, row, annotation_matrix[row][col],
                    ha="center", va="center", fontsize=6,
                    color="white", fontweight="bold")

    ax.set_xlabel("Token")
    ax.set_ylabel("MoE Layer")
    ax.set_title(f"Token-Level Expert Routing -- {prompt_name}\n"
                 f"(cell = top-1 expert, parentheses = top-2 expert)")
    ax.set_xticks(range(num_tokens))
    truncated = [t.replace("\n", "\\n")[:10] for t in tokens]
    ax.set_xticklabels(truncated, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(num_layers))
    ax.set_yticklabels([f"L{idx}" for idx in layer_indices], fontsize=8)
    plt.colorbar(im, ax=ax, label="Expert ID", shrink=0.7)

    plt.tight_layout()
    plt.show()

In [ ]:
# Detailed token-level routing for the creative prompt
plot_token_routing_detail(
    all_routing_data["creative"],
    all_tokens["creative"],
    "creative",
)

In [ ]:
# Detailed token-level routing for the code prompt
plot_token_routing_detail(
    all_routing_data["code"],
    all_tokens["code"],
    "code",
)

In [ ]:
# Detailed token-level routing for the math prompt
plot_token_routing_detail(
    all_routing_data["math"],
    all_tokens["math"],
    "math",
)

---

## 8. Expert Similarity Analysis

We compare router decisions between different prompts to understand whether
different input types activate systematically different sets of experts.
We compute cosine similarity between the average router probability vectors
across prompts.

In [ ]:
def compute_avg_routing_profile(routing_data, num_experts):
    """
    Compute the average router probability vector across all layers and tokens
    for a single prompt. Returns a (num_experts,) vector.
    """
    all_probs = []
    for layer_idx, data in routing_data.items():
        # data["probs"]: (1, seq_len, num_experts)
        probs = data["probs"][0].numpy()  # (seq_len, num_experts)
        all_probs.append(probs)
    # Stack all (layer, token) probability vectors
    stacked = np.concatenate(all_probs, axis=0)  # (total_positions, num_experts)
    return stacked.mean(axis=0)  # (num_experts,)


def compute_per_layer_routing_profile(routing_data, num_experts):
    """
    Compute average router probability per layer.
    Returns a (num_moe_layers, num_experts) matrix.
    """
    layer_indices = sorted(routing_data.keys())
    profiles = np.zeros((len(layer_indices), num_experts))
    for row, layer_idx in enumerate(layer_indices):
        probs = routing_data[layer_idx]["probs"][0].numpy()
        profiles[row] = probs.mean(axis=0)
    return profiles, layer_indices


# Compute average routing profiles
prompt_names = list(prompts.keys())
profiles = {}
for pname in prompt_names:
    profiles[pname] = compute_avg_routing_profile(
        all_routing_data[pname], num_experts_total
    )
    print(f"{pname:>12}: top-3 experts by avg probability: "
          f"{np.argsort(profiles[pname])[-3:][::-1].tolist()}")

In [ ]:
# Cosine similarity between routing profiles
profile_matrix = np.stack([profiles[pname] for pname in prompt_names])
sim_matrix = cosine_similarity(profile_matrix)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim_matrix, cmap="RdYlGn", vmin=0.5, vmax=1.0)

# Annotate cells
for i in range(len(prompt_names)):
    for j in range(len(prompt_names)):
        ax.text(j, i, f"{sim_matrix[i, j]:.3f}",
                ha="center", va="center", fontsize=11,
                fontweight="bold" if i == j else "normal")

ax.set_xticks(range(len(prompt_names)))
ax.set_xticklabels(prompt_names, rotation=30, ha="right")
ax.set_yticks(range(len(prompt_names)))
ax.set_yticklabels(prompt_names)
ax.set_title("Cosine Similarity of Router Probability Profiles")
plt.colorbar(im, ax=ax, label="Cosine similarity", shrink=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Per-layer routing profile comparison: math vs code vs creative
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

comparisons = [
    ("math", "code"),
    ("math", "creative"),
    ("code", "creative"),
]

for ax, (pname_a, pname_b) in zip(axes, comparisons):
    profile_a, layers_a = compute_per_layer_routing_profile(
        all_routing_data[pname_a], num_experts_total
    )
    profile_b, layers_b = compute_per_layer_routing_profile(
        all_routing_data[pname_b], num_experts_total
    )

    # Compute per-layer cosine similarity
    num_comparable = min(len(layers_a), len(layers_b))
    layer_sims = []
    layer_labels = []
    for i in range(num_comparable):
        sim = cosine_similarity(profile_a[i:i+1], profile_b[i:i+1])[0, 0]
        layer_sims.append(sim)
        layer_labels.append(layers_a[i])

    ax.plot(range(num_comparable), layer_sims, "o-", markersize=3, linewidth=1.5)
    ax.axhline(y=1.0, color="gray", linestyle=":", alpha=0.5)
    ax.set_xlabel("MoE layer (index in sequence)")
    ax.set_ylabel("Cosine similarity")
    ax.set_title(f"{pname_a} vs {pname_b}")
    ax.set_ylim(0, 1.05)

plt.suptitle("Per-Layer Router Similarity Between Prompt Types", fontsize=13,
             fontweight="bold")
plt.tight_layout()
plt.show()

---

## 9. Routing Entropy per Layer

The entropy of the router's probability distribution tells us how "decisive" the
router is at each layer:

- **Low entropy**: the router strongly prefers specific experts (confident routing)
- **High entropy**: the router distributes probability more evenly (uncertain routing)

Maximum entropy for a uniform distribution over $N$ experts is $\log_2(N)$.

In [ ]:
def compute_routing_entropy(routing_data, num_experts):
    """
    Compute the average entropy of the router distribution at each MoE layer.
    Returns:
        layer_indices: list of layer indices
        mean_entropies: (num_layers,) array of mean entropy values
        std_entropies: (num_layers,) array of entropy std across tokens
    """
    layer_indices = sorted(routing_data.keys())
    mean_entropies = []
    std_entropies = []

    for layer_idx in layer_indices:
        probs = routing_data[layer_idx]["probs"][0].numpy()  # (seq_len, num_experts)
        # Compute entropy for each token position (base 2 for bits)
        token_entropies = scipy_entropy(probs.T, base=2)  # (seq_len,)
        mean_entropies.append(np.mean(token_entropies))
        std_entropies.append(np.std(token_entropies))

    return layer_indices, np.array(mean_entropies), np.array(std_entropies)

In [ ]:
max_entropy = np.log2(num_experts_total)

fig, ax = plt.subplots(figsize=(14, 6))

prompt_colors_map = {
    "math": "#e74c3c",
    "code": "#3498db",
    "creative": "#2ecc71",
    "factual": "#f39c12",
}

for pname in prompt_names:
    layers, means, stds = compute_routing_entropy(
        all_routing_data[pname], num_experts_total
    )
    x = range(len(layers))
    color = prompt_colors_map[pname]
    ax.plot(x, means, "o-", label=pname, color=color, markersize=3, linewidth=1.5)
    ax.fill_between(x, means - stds, means + stds, alpha=0.15, color=color)

# Reference lines
ax.axhline(y=max_entropy, color="gray", linestyle="--", alpha=0.5,
           label=f"Max entropy (uniform over {num_experts_total} experts) = {max_entropy:.2f} bits")
ax.axhline(y=0, color="gray", linestyle=":", alpha=0.3)

ax.set_xlabel("MoE layer (index in sequence)")
ax.set_ylabel("Entropy (bits)")
ax.set_title("Routing Entropy Across MoE Layers")
ax.legend(loc="best", fontsize=9)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

print(f"Maximum possible entropy (uniform over {num_experts_total} experts): {max_entropy:.3f} bits")
print()
for pname in prompt_names:
    layers, means, stds = compute_routing_entropy(
        all_routing_data[pname], num_experts_total
    )
    print(f"{pname:>12}: mean entropy = {means.mean():.3f} bits "
          f"({means.mean() / max_entropy * 100:.1f}% of max), "
          f"range [{means.min():.3f}, {means.max():.3f}]")

In [ ]:
# Entropy heatmap: layer x token for each prompt
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, pname in zip(axes.flat, prompt_names):
    routing_data = all_routing_data[pname]
    layer_indices = sorted(routing_data.keys())
    tokens = all_tokens[pname]
    num_tokens = len(tokens)

    # Sample layers if too many
    if len(layer_indices) > 20:
        step = max(1, len(layer_indices) // 20)
        sampled_layers = layer_indices[::step]
    else:
        sampled_layers = layer_indices

    entropy_matrix = np.zeros((len(sampled_layers), num_tokens))
    for row, layer_idx in enumerate(sampled_layers):
        probs = routing_data[layer_idx]["probs"][0].numpy()  # (seq_len, num_experts)
        for tok in range(num_tokens):
            entropy_matrix[row, tok] = scipy_entropy(probs[tok], base=2)

    im = ax.imshow(entropy_matrix, aspect="auto", cmap="magma_r",
                    vmin=0, vmax=max_entropy, interpolation="nearest")
    ax.set_xlabel("Token")
    ax.set_ylabel("MoE Layer")
    ax.set_title(f"{pname}: \"{prompts[pname][:35]}...\"" if len(prompts[pname]) > 35
                 else f"{pname}: \"{prompts[pname]}\"")
    truncated = [t[:6] for t in tokens]
    ax.set_xticks(range(num_tokens))
    ax.set_xticklabels(truncated, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(len(sampled_layers)))
    ax.set_yticklabels(sampled_layers, fontsize=6)
    plt.colorbar(im, ax=ax, label="Entropy (bits)", shrink=0.7)

plt.suptitle("Router Entropy: Layer x Token", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---

## Summary

In this notebook we loaded a real Mixture of Experts model and visualized its
expert routing behavior in action:

1. **Model Loading** -- loaded Qwen1.5-MoE-A2.7B-Chat (or Mixtral fallback)
   and inspected its MoE architecture.
2. **Architecture Inspection** -- enumerated MoE layers, gate shapes, number
   of experts, and active expert count per token.
3. **Router Hooks** -- registered forward hooks to intercept routing decisions
   (logits, probabilities, and top-k selections).
4. **Inference Capture** -- ran four different prompt types (math, code,
   creative, factual) and captured all routing data.
5. **Activation Heatmaps** -- visualized which experts "light up" for each
   token at each layer, for both top-1 and top-2 selections.
6. **Expert Utilization** -- measured how evenly experts are used across
   prompts and layers.
7. **Token-Level Routing** -- created annotated maps showing exactly which
   expert handles each token at each layer.
8. **Similarity Analysis** -- compared routing patterns between prompt types
   using cosine similarity, revealing how different input domains activate
   different expert subsets.
9. **Routing Entropy** -- measured how decisive the router is at each layer,
   showing that some layers route confidently while others are more uncertain.

In [ ]:
# Clean up
capture.remove_hooks()
print("Hooks removed. Done.")